# A3 · Midterm Review in Code: Chapters 1–7

**IMSE 774 · Neural Networks · Formative Assignment 3 · Weeks 1–7**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harunpirim/IMSE774-NeuralNetworks/blob/main/assignments/A3-midterm-review.ipynb)

The midterm on **Thursday, October 22** is closed-book and multiple-choice, and it tests whether you understand the ideas of Chapters 1–7 — not whether you can type them. This assignment is the other half of the same preparation: every task here makes one of those ideas concrete in code, and every question on the midterm asks about an idea you will have built here. If you can explain *why* each check below passes, you are ready.

- **Weeks 1–2 — [Introduction](https://harunpirim.github.io/IMSE774-NeuralNetworks/notes/01-introduction.html) and [Background](https://harunpirim.github.io/IMSE774-NeuralNetworks/notes/02-math-background.html)**: Prince, Chapters 1–2 and Appendix B.
- **Weeks 3–4 — [Shallow](https://harunpirim.github.io/IMSE774-NeuralNetworks/notes/03-shallow-networks.html) and [Deep Networks](https://harunpirim.github.io/IMSE774-NeuralNetworks/notes/04-deep-networks.html)**: Chapters 3–4.
- **Week 5 — [Loss Functions](https://harunpirim.github.io/IMSE774-NeuralNetworks/notes/05-loss-functions.html)**: Chapter 5.
- **Week 6 — [Fitting Models](https://harunpirim.github.io/IMSE774-NeuralNetworks/notes/06-fitting-models.html)**: Chapter 6.
- **Week 7 — [Gradients and Initialization](https://harunpirim.github.io/IMSE774-NeuralNetworks/notes/07-gradients-initialization.html)**: Chapter 7.

### What "formative" means here

As in A1 and A2, the purpose is feedback, not ranking. Chapters 1–4 were the subject of the first two assignments, so Part A revisits them quickly and the weight is on Chapters 5–7. Almost every task ends in a **self-check**; the written prompts ask for the reasoning, and the reasoning is what the midterm will ask for. An honest wrong answer with your reasoning is more useful to both of us than a right answer you cannot explain.

### How to work

1. **File → Save a copy in Drive**, so your edits persist.
2. Run the **Setup** cell first. Every later cell depends on the names it defines.
3. Work top to bottom. Blanks look like `None`, `...`, or `# YOUR CODE HERE`. Written prompts have a cell that starts with *Your answer:* — write directly in it (double-click to edit).
4. Run the check cell under each task. ✅ means the idea landed; ❌ comes with a hint that says where to look; ⬜ means you have not answered yet.
5. When you are done, **Runtime → Run all**, confirm there are no errors, and run the **Summary** at the bottom.
6. **Submit on Blackboard under Assignment A3.** Download your notebook with **File → Download → Download .ipynb** and upload that file, named `A3_<lastname>.ipynb`. Do not submit a link to your Drive copy. The deadline is posted on Blackboard; plan to finish before the midterm.

Budget about **5 hours**. Times next to each task are rough. You may use any resource, including the course notebooks and the textbook, but write your own code and your own sentences. Each task names the midterm concept it prepares.

| Part | Covers | Tasks |
|:--|:--|:--|
| **A** | Chapters 1–4 in brief: the supervised recipe, generalization, what a network computes, depth versus width | A1–A3 |
| **B** | Chapter 5: the maximum likelihood recipe, least squares, binary and multiclass cross-entropy, logits | B1–B4 |
| **C** | Chapter 6: gradient descent and its step size, non-convexity, SGD, momentum, Adam | C1–C4 |
| **D** | Chapter 7: backpropagation by hand and in matrix form, its cost, initialization | D1–D4 |
| **E** | Self-assessment | E1 |

## Setup

Run this first. It imports the libraries, fixes the random seeds, and defines the self-check helpers.

In [ ]:
import hashlib
import time
from math import comb
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

torch.manual_seed(774)
np.random.seed(774)
rng = np.random.default_rng(774)

plt.rcParams.update({
    "figure.figsize": (7, 4),
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "legend.frameon": False,
})

NDSU_GREEN = "#005643"
NDSU_GOLD  = "#FFC72C"
ACCENT     = "#C1121F"
STEEL      = "#1864AB"

# ---------------------------------------------------------------------------
# Self-check helpers. A failed check is feedback, never a crash.
# ---------------------------------------------------------------------------
_RESULTS = {}

def _key(s):
    return " ".join(str(s).strip().lower().replace("_", " ").split())

def _fingerprint(s):
    return hashlib.sha256(_key(s).encode()).hexdigest()[:12]

def check(label, ok, hint=""):
    """Record and print one pass/fail line. Pass ok=None for 'not answered yet'."""
    if ok is None:
        _RESULTS[label] = False
        print("⬜ " + label + "   ← not answered yet")
        return False
    ok = bool(ok)
    _RESULTS[label] = ok
    print(("✅ " if ok else "❌ ") + label + ("" if ok or not hint else "   ← " + hint))
    return ok

def _join(*parts):
    """'a|b|c' for the fingerprint checks, or None if any part is still blank."""
    return None if any(p is None for p in parts) else "|".join(str(p) for p in parts)

def check_answer(label, given, fingerprint, hint=""):
    """Compare a short written answer with a fingerprint of the expected one.
    Case, spacing, and underscores are ignored."""
    if given is None or (isinstance(given, str) and not given.strip()):
        _RESULTS[label] = False
        print("⬜ " + label + "   ← not answered yet")
        return False
    return check(label, _fingerprint(given) == fingerprint, hint)

def check_close(label, got, want, tol=1e-3, hint=""):
    """Numerical comparison with a relative and absolute tolerance."""
    if got is None:
        _RESULTS[label] = False
        print("⬜ " + label + "   ← not answered yet")
        return False
    try:
        ok = np.allclose(np.asarray(got, dtype=float), np.asarray(want, dtype=float),
                         rtol=tol, atol=tol)
    except Exception:
        ok = False
    return check(label, ok, hint)

def summary():
    parts = {}
    for label, ok in _RESULTS.items():
        part = label[0]
        parts.setdefault(part, [0, 0])
        parts[part][0] += int(ok)
        parts[part][1] += 1
    print(f"{'part':<6}{'passed':>8}{'of':>6}")
    print("-" * 20)
    for part in sorted(parts):
        p, n = parts[part]
        print(f"{part:<6}{p:>8}{n:>6}")
    p = sum(v[0] for v in parts.values()); n = sum(v[1] for v in parts.values())
    print("-" * 20)
    print(f"{'all':<6}{p:>8}{n:>6}")
    missing = [k for k, v in _RESULTS.items() if not v]
    if missing:
        print("\nNot yet passing:", ", ".join(missing))

print("Setup complete. torch", torch.__version__, "· numpy", np.__version__)

## Part A — Chapters 1–4 in brief

*Weeks 1–4 notes; Prince, Chapters 1–4.*

Three tasks, one idea each: what training does and what it cannot tell you; what a ReLU network computes; and why depth is not the same as width. A1 and A2 covered these chapters in detail — this is the part of them the midterm leans on.

### A1 · The recipe, and why training loss cannot choose the model *(25 min)*

*Midterm: the supervised learning recipe; generalization; under- and overfitting (Prince §2.2–2.3).*

Twelve noisy samples of a smooth function. You will fit polynomials of degree 1, 3, and 11 by least squares — a **model** $\mathrm{f}[x, \boldsymbol{\phi}]$ that is linear in its parameters, so the **loss** $L[\boldsymbol{\phi}] = \sum_i (\mathrm{f}[x_i, \boldsymbol{\phi}] - y_i)^2$ has a closed-form minimizer — and then measure the loss on **test** data generated the same way.

1. Implement `poly_features` so that column $j$ holds $x^j$, and `fit_poly` with `np.linalg.lstsq`.
2. Fill the table of training and test losses (mean squared error per example).
3. Name the degree that underfits, the one that overfits, and the one you would deploy.

In [ ]:
def true_f(x):
    return np.sin(2.5 * x) + 0.3 * x

x_train = np.sort(rng.uniform(-1, 1, 12))
y_train = true_f(x_train) + rng.normal(0, 0.15, 12)
x_test  = np.sort(rng.uniform(-1, 1, 200))
y_test  = true_f(x_test) + rng.normal(0, 0.15, 200)

def poly_features(x, degree):
    """Design matrix with columns x^0, x^1, ..., x^degree; shape (len(x), degree + 1)."""
    return None                                     # YOUR CODE HERE

def fit_poly(x, y, degree):
    """Least squares coefficients phi (length degree + 1)."""
    return None                                     # YOUR CODE HERE: np.linalg.lstsq(..., rcond=None)[0]

def mse(phi, x, y, degree):
    return None                                     # YOUR CODE HERE: mean of squared residuals

losses_A1 = {}                                      # degree -> (train MSE, test MSE)
for d in (1, 3, 11):
    pass                                            # YOUR CODE HERE

print(f"{'degree':>7}{'train MSE':>12}{'test MSE':>12}")
for d, (tr, te) in losses_A1.items():
    print(f"{d:>7}{tr:>12.4f}{te:>12.4f}")

underfits = None                                    # a degree
overfits  = None                                    # a degree
deploy    = None                                    # a degree

xg = np.linspace(-1, 1, 300)
plt.figure(figsize=(6.5, 3.4))
plt.plot(xg, true_f(xg), "k--", lw=1, label="true function")
plt.plot(x_train, y_train, "o", color=NDSU_GOLD, mec="k", label="training data")
for d, c in zip((1, 3, 11), (STEEL, NDSU_GREEN, ACCENT)):
    phi = fit_poly(x_train, y_train, d)
    if phi is not None:
        plt.plot(xg, poly_features(xg, d) @ phi, color=c, label=f"degree {d}")
plt.ylim(-2, 2); plt.legend(ncol=2, fontsize=8); plt.show()

In [ ]:
_V = lambda x, d: np.vander(np.asarray(x, float), d + 1, increasing=True)
_ref = {}
for d in (1, 3, 11):
    _phi = np.linalg.lstsq(_V(x_train, d), y_train, rcond=None)[0]
    _ref[d] = (np.mean((_V(x_train, d) @ _phi - y_train) ** 2), np.mean((_V(x_test, d) @ _phi - y_test) ** 2))

check("A1 poly_features shape and columns",
      None if poly_features(x_train, 3) is None else np.allclose(poly_features(x_train, 3), _V(x_train, 3)),
      "Column j must be x**j, starting with a column of ones (j = 0).")
check_close("A1 train/test losses", None if len(losses_A1) < 3 else [losses_A1[d] for d in (1, 3, 11)],
            [_ref[d] for d in (1, 3, 11)], tol=1e-6,
            hint="MSE is the mean over examples of (prediction - y)**2; fit on training data only.")
_tr = {d: _ref[d][0] for d in _ref}; _te = {d: _ref[d][1] for d in _ref}
check("A1 training loss falls monotonically with degree", _tr[1] > _tr[3] > _tr[11])
check_answer("A1 which degree underfits", underfits, _fingerprint(max(_te, key=lambda d: _tr[d])),
             "High training AND high test loss: the model cannot represent the function.")
check_answer("A1 which degree overfits", overfits, _fingerprint(min(_tr, key=_tr.get)),
             "Lowest training loss but a test loss far above it.")
check_answer("A1 which degree to deploy", deploy, _fingerprint(min(_te, key=_te.get)),
             "The one with the lowest TEST loss.")

**Written.** The degree-11 polynomial has the lowest training loss of the three, and it is the worst model. In two or three sentences: what does the training loss measure, what does the test loss measure, and why can the first never be used to choose between these three models? Use the words *generalization* and *overfitting*.

*Your answer:*

### A2 · What a ReLU network computes *(25 min)*

*Midterm: pre-activations, activations, joints, linear regions, why the activation must be nonlinear (Prince §3.1–3.4).*

A shallow network with one input, $D$ hidden ReLU units, and one output is $y = \phi_0 + \sum_d \phi_d\,\mathrm{ReLU}[\theta_{d0} + \theta_{d1}x]$. Its output is piecewise linear; the joints are where pre-activations cross zero; within a region the slope is the sum of $\theta_{d1}\phi_d$ over the **active** units.

1. Implement `shallow` returning the output and the matrix of pre-activations.
2. Implement `realized_regions`, which counts the distinct activation patterns along a fine grid — the number of linear regions the network *actually* has.
3. Replace the ReLU by the identity and count again.

In [ ]:
D_A2 = 6
theta_A2 = rng.normal(0, 1, (D_A2, 2))             # theta[d] = (theta_d0, theta_d1)
phi0_A2  = 0.1
phi_A2   = rng.normal(0, 1, D_A2)

def shallow(x, theta, phi0, phi, act=lambda z: np.maximum(z, 0)):
    """Returns (y, pre). pre has shape (D, len(x)): pre[d] = theta_d0 + theta_d1 * x."""
    x = np.asarray(x, float)
    pre = None                                      # YOUR CODE HERE
    y = None                                        # YOUR CODE HERE: phi0 + phi @ act(pre)
    return y, pre

def realized_regions(pre):
    """Number of distinct on/off patterns (columns of pre > 0) along the grid."""
    return None                                     # YOUR CODE HERE: hint, np.unique(..., axis=1)

xg = np.linspace(-4, 4, 20001)
y_relu, pre = shallow(xg, theta_A2, phi0_A2, phi_A2)
y_lin, _    = shallow(xg, theta_A2, phi0_A2, phi_A2, act=lambda z: z)

regions_relu = None if pre is None else realized_regions(pre)
joints_A2 = None                                    # YOUR CODE HERE: the D joints, sorted (from theta_A2)
max_regions_A2 = None                               # by hand: the most regions a shallow network with D_A2 units and one input can have
regions_identity = None                             # by hand: how many linear regions does y_lin have?

if y_relu is not None:
    plt.figure(figsize=(6.5, 3.2))
    plt.plot(xg, y_relu, color=NDSU_GREEN, lw=2, label="ReLU")
    plt.plot(xg, y_lin, color=ACCENT, lw=1.2, label="identity activation")
    if joints_A2 is not None:
        for j in joints_A2: plt.axvline(j, color=NDSU_GOLD, lw=1)
    plt.legend(); plt.xlabel("$x$"); plt.ylabel("$y$"); plt.show()
print("realized regions, ReLU:", regions_relu)

In [ ]:
_pre_ref = theta_A2[:, 0:1] + theta_A2[:, 1:2] * xg[None, :]
check_close("A2 shallow output", y_relu, phi0_A2 + phi_A2 @ np.maximum(_pre_ref, 0), tol=1e-9,
            hint="pre = theta[:, 0:1] + theta[:, 1:2] * x[None, :]; y = phi0 + phi @ act(pre).")
check_close("A2 realized regions", regions_relu, np.unique(_pre_ref > 0, axis=1).shape[1], tol=0,
            hint="Count the distinct columns of (pre > 0); every joint inside the grid adds one region.")
check_close("A2 joints", joints_A2, np.sort(-theta_A2[:, 0] / theta_A2[:, 1]), tol=1e-6,
            hint="A joint is where theta_d0 + theta_d1 x = 0.")
check_close("A2 maximum regions for a shallow 1-D network", max_regions_A2, D_A2 + 1, tol=0,
            hint="D joints split the line into D + 1 pieces.")
check_close("A2 regions with the identity activation", regions_identity, 1, tol=0,
            hint="Look at the red curve. A linear function of a linear function is linear.")
check("A2 identity output is a straight line",
      None if y_lin is None else np.allclose(np.diff(y_lin, 2), 0, atol=1e-8))

**Written.** The red curve is the same network with $a[z] = z$. Explain, in terms of matrix algebra, why *any* number of hidden units then buys nothing — write the composed map in one line. Then state in one sentence what the nonlinearity contributes that makes universal approximation possible.

*Your answer:*

### A3 · Depth multiplies, width adds *(25 min)*

*Midterm: parameter counting, maximum regions of shallow versus deep networks, representability versus trainability (Prince §4.4–4.5).*

For one input and one output, a network with $K$ hidden layers of $D$ units each has $3D + 1 + (K-1)D(D+1)$ parameters and can have at most $(D+1)^K$ linear regions; a shallow network with $D$ units has $3D + 1$ parameters and at most $D + 1$ regions.

1. Implement both counts.
2. Build the deep network in PyTorch with `make_mlp`, confirm the parameter count against `sum(p.numel() ...)`, and count its **realized** regions on a grid at random initialization.
3. Compare the realized count with the maximum.

In [ ]:
def n_params(D, K):
    """Parameters of a 1-D network with K hidden layers of width D (K = 1 is shallow)."""
    return None                                     # YOUR CODE HERE

def max_regions(D, K):
    return None                                     # YOUR CODE HERE

def make_mlp(widths):
    layers, d_in = [], 1
    for w in widths:
        layers += [nn.Linear(d_in, w), nn.ReLU()]
        d_in = w
    layers.append(nn.Linear(d_in, 1))
    return nn.Sequential(*layers)

def realized_regions_torch(model, x):
    """Distinct patterns of which ReLUs are active, over the inputs x (shape (n, 1))."""
    pats = []
    h = x
    with torch.no_grad():
        for layer in model:
            h = layer(h)
            if isinstance(layer, nn.ReLU):
                pats.append(h > 0)
    pats = torch.cat(pats, dim=1)
    return None                                     # YOUR CODE HERE: number of distinct rows of pats

torch.manual_seed(774)
D_A3, K_A3 = 10, 4
mlp = make_mlp([D_A3] * K_A3)
xg_t = torch.linspace(-3, 3, 100001)[:, None]

params_formula = n_params(D_A3, K_A3)
params_torch   = sum(p.numel() for p in mlp.parameters())
regions_max    = max_regions(D_A3, K_A3)
regions_real   = realized_regions_torch(mlp, xg_t)
shallow_match  = None                               # YOUR CODE HERE: smallest shallow width D with n_params(D, 1) >= params_formula

print("parameters: formula", params_formula, " torch", params_torch)
print("regions:    maximum", regions_max, " realized at initialization", regions_real)
print("a shallow network with the same budget has width", shallow_match, "and at most", None if shallow_match is None else shallow_match + 1, "regions")

In [ ]:
check_close("A3 parameter count matches PyTorch", params_formula, params_torch, tol=0,
            hint="Each Linear(a, b) has a*b weights and b biases; add them up for every layer.")
check_close("A3 maximum regions", regions_max, (D_A3 + 1) ** K_A3, tol=0, hint="(D + 1) ** K.")
check("A3 realized regions counted", None if regions_real is None else 1 < regions_real < regions_max,
      "torch.unique(pats, dim=0) gives the distinct activation patterns.")
_pf = 3 * D_A3 + 1 + (K_A3 - 1) * D_A3 * (D_A3 + 1)
check_close("A3 shallow network with the same parameter budget", shallow_match,
            next(D for D in range(1, 10000) if 3 * D + 1 >= _pf), tol=0,
            hint="Smallest D with 3D + 1 >= the deep count.")

**Written.** The maximum region count is in the thousands, the realized count at initialization is a few dozen, and a shallow network with the same parameter budget could have about a hundred and twenty. In three sentences: what kind of statement is $(D+1)^K$ (what does it say and not say), which of Prince's claims about deep versus shallow networks is a *theorem* and which are *observations*, and why is the realized count so far below the maximum?

*Your answer:*

## Part B — Loss functions

*Week 5 note; Prince, Chapter 5.*

Chapter 5 has one recipe. The network does not predict $y$; it predicts the parameters $\boldsymbol{\theta} = \mathrm{f}[\mathbf{x}, \boldsymbol{\phi}]$ of a distribution $Pr(y \mid \boldsymbol{\theta})$. The loss is the negative log-likelihood of the training labels under those distributions, $L = -\sum_i \log Pr(y_i \mid \mathrm{f}[\mathbf{x}_i, \boldsymbol{\phi}])$. Everything in this part is that recipe applied three times.

### B1 · Why the product becomes a sum *(15 min)*

*Midterm: maximum likelihood, log-likelihood, negative log-likelihood, why logs (Prince §5.1).*

Suppose a model assigns probability density $Pr(y_i \mid \mathrm{f}[x_i, \boldsymbol{\phi}])$ to each of $I$ independent training labels. The likelihood of the whole training set is the **product** of these; maximizing it is the same as maximizing the **sum of logs**, which is the same as minimizing the **negative** sum of logs.

1. Compute the product and the sum-of-logs for growing $I$, using the normal densities in `probs`.
2. Report the first $I$ at which the product underflows to exactly `0.0` in double precision while the sum of logs is still a perfectly ordinary number.

In [ ]:
resid = rng.normal(0, 1, 5000)
probs = np.exp(-0.5 * resid ** 2) / np.sqrt(2 * np.pi)   # Pr(y_i | f[x_i, phi]) for a unit-variance normal

def likelihood(p):      return None                       # YOUR CODE HERE: the product, equation (5.1)
def log_likelihood(p):  return None                       # YOUR CODE HERE: the sum of logs, equation (5.3)

print(f"{'I':>6}{'product':>16}{'sum of logs':>16}")
for I in (10, 100, 500, 1000, 5000):
    if likelihood(probs[:I]) is not None and log_likelihood(probs[:I]) is not None:
        print(f"{I:>6}{likelihood(probs[:I]):>16.3e}{log_likelihood(probs[:I]):>16.2f}")

first_underflow = None                                    # YOUR CODE HERE: smallest I with likelihood(probs[:I]) == 0.0
smallest_factor = probs.min()
print("first I whose product is exactly 0.0:", first_underflow, "   smallest single factor:", f"{smallest_factor:.3e}")

In [ ]:
check_close("B1 likelihood is the product", likelihood(probs[:50]), np.prod(probs[:50]), tol=1e-9, hint="np.prod.")
check_close("B1 log-likelihood is the sum of logs", log_likelihood(probs[:50]), np.log(probs[:50]).sum(), tol=1e-9, hint="np.sum(np.log(p)).")
_fu = next(I for I in range(1, 5001) if np.prod(probs[:I]) == 0.0)
check_close("B1 first underflow", first_underflow, _fu, tol=0, hint="Loop over I and stop at the first exact zero.")
check("B1 the sum of logs is still finite there", np.isfinite(np.log(probs[:_fu]).sum()))

**Written.** None of the individual probabilities is remotely close to zero, yet their product is. State the two reasons Prince gives for working with the log-likelihood rather than the likelihood, and explain why the position of the maximum is unchanged by taking the log.

*Your answer:*

### B2 · Least squares is a normal distribution in disguise *(25 min)*

*Midterm: the recipe for univariate regression; what least squares assumes; why $\sigma^2$ drops out (Prince §5.3).*

Choose $Pr(y \mid \mu, \sigma^2)$ normal with mean $\mu = \mathrm{f}[x, \boldsymbol{\phi}]$ and fixed variance. The negative log-likelihood is
$$L = \sum_i \left[\tfrac{1}{2}\log(2\pi\sigma^2) + \frac{(y_i - \mathrm{f}[x_i, \boldsymbol{\phi}])^2}{2\sigma^2}\right],$$
and the terms that do not depend on $\boldsymbol{\phi}$ can be dropped, leaving least squares.

1. Implement `nll_normal` for a line $\mathrm{f}[x, \boldsymbol{\phi}] = \phi_0 + \phi_1 x$ and `sse`, the sum of squared errors.
2. Minimize both over a grid of slopes for three values of $\sigma^2$ and confirm the minimizers agree.
3. Then do the same with a *Laplace* distribution, $Pr(y \mid \mu) \propto \exp(-|y - \mu|/b)$, and see what loss that recipe produces.

In [ ]:
x_b = np.linspace(0, 2, 15)
y_b = 0.8 + 0.5 * x_b + rng.normal(0, 0.2, 15)
y_b[-1] += 2.5                                            # one gross outlier
slopes = np.linspace(-1, 2, 3001)

def sse(slope, intercept=0.8):
    return None                                           # YOUR CODE HERE: sum_i (y_i - (intercept + slope x_i))^2

def nll_normal(slope, sigma2, intercept=0.8):
    """Full negative log-likelihood of a normal with mean intercept + slope*x, including the constant terms."""
    return None                                           # YOUR CODE HERE

def nll_laplace(slope, b=0.2, intercept=0.8):
    """NLL of a Laplace distribution, density exp(-|y - mu| / b) / (2b)."""
    return None                                           # YOUR CODE HERE

argmin_sse = None                                         # YOUR CODE HERE: slope minimizing sse over the grid
argmin_nll = {}                                           # sigma2 -> minimizing slope
for s2 in (0.05, 0.5, 5.0):
    pass                                                  # YOUR CODE HERE
argmin_laplace = None                                     # YOUR CODE HERE

print("least squares slope:", argmin_sse)
print("normal NLL slopes:  ", argmin_nll)
print("Laplace NLL slope:  ", argmin_laplace)
least_squares_assumes = None                              # "normal" or "laplace": which noise model is least squares?
laplace_loss_name = None                                  # the loss the Laplace recipe produces: "absolute error" or "squared error"

In [ ]:
_r = lambda s: y_b - (0.8 + s * x_b)
_sse_ref = slopes[np.argmin([np.sum(_r(s) ** 2) for s in slopes])]
_lap_ref = slopes[np.argmin([np.sum(np.abs(_r(s))) for s in slopes])]
check_close("B2 sse", sse(0.3), np.sum(_r(0.3) ** 2), tol=1e-9)
check_close("B2 nll_normal includes the constant", nll_normal(0.3, 0.5),
            np.sum(0.5 * np.log(2 * np.pi * 0.5) + _r(0.3) ** 2), tol=1e-6,
            hint="Each term is 0.5*log(2 pi sigma2) + r^2 / (2 sigma2).")
check_close("B2 least squares minimizer", argmin_sse, _sse_ref, tol=1e-9)
check("B2 the normal NLL minimizer does not depend on sigma^2",
      None if len(argmin_nll) < 3 else all(abs(v - _sse_ref) < 1e-9 for v in argmin_nll.values()),
      "Changing sigma2 rescales the loss and shifts it by a constant; neither moves the minimum.")
check_close("B2 Laplace minimizer", argmin_laplace, _lap_ref, tol=1e-9,
            hint="The Laplace NLL is sum |r| / b plus a constant: minimizing absolute error.")
check("B2 the Laplace fit is less pulled by the outlier", abs(_lap_ref - 0.5) < abs(_sse_ref - 0.5))
check_answer("B2 least squares assumes which noise", least_squares_assumes, '317b32c14369')
check_answer("B2 the loss the Laplace recipe produces", laplace_loss_name, 'ad0c2cd42a82')

**Written.** Prince's recipe produced least squares from one choice and absolute error from another. In two or three sentences: what assumption about the data does least squares encode, and what does the outlier experiment say about it? Then explain why the variance $\sigma^2$ never appears in the least squares loss even though the model has one.

*Your answer:*

### B3 · Binary classification and the logit trap *(30 min)*

*Midterm: the Bernoulli construction, the sigmoid, binary cross-entropy, why losses take logits (Prince §5.4; Week 5 "three rules").*

The network output $z = \mathrm{f}[x, \boldsymbol{\phi}]$ is a real number — a **logit**. The sigmoid $\lambda = \mathrm{sig}[z] = 1/(1 + e^{-z})$ maps it to a Bernoulli parameter, and the negative log-likelihood is the **binary cross-entropy**
$$L = -\sum_i \big[(1 - y_i)\log(1 - \lambda_i) + y_i \log \lambda_i\big].$$

1. Implement `sigmoid` and `bce_from_logits` and match `nn.BCEWithLogitsLoss(reduction="sum")`.
2. Show algebraically (in code) that $-\log \mathrm{sig}[z] = \log(1 + e^{-z})$, and implement it stably with `np.logaddexp`.
3. Reproduce the logit trap: evaluate `nn.BCELoss()(torch.sigmoid(z), 1)` and `nn.BCEWithLogitsLoss()(z, 1)` at very negative $z$ and compare their gradients.

In [ ]:
def sigmoid(z):
    return None                                           # YOUR CODE HERE

def bce_from_logits(z, y):
    """Binary cross-entropy summed over examples, from logits z and labels y in {0, 1}. Equation (5.20)."""
    lam = sigmoid(z)
    return None                                           # YOUR CODE HERE

def neg_log_sig_stable(z):
    """-log sig[z] = log(1 + exp(-z)), computed without overflow."""
    return None                                           # YOUR CODE HERE: np.logaddexp(0, -z)

torch.manual_seed(774)
z_t = torch.randn(10); y_t = (torch.rand(10) < 0.5).float()
ours  = bce_from_logits(z_t.numpy(), y_t.numpy())
torch_val = nn.BCEWithLogitsLoss(reduction="sum")(z_t, y_t).item()
print(f"ours {ours}   PyTorch {torch_val:.6f}")

rows = []
for zv in (-5.0, -20.0, -100.0, -500.0):
    ta = torch.tensor([zv], requires_grad=True); tb = torch.tensor([zv], requires_grad=True)
    one = torch.tensor([1.0])
    La = nn.BCELoss()(torch.sigmoid(ta), one)             # the trap: sigmoid first, then a loss on probabilities
    Lb = nn.BCEWithLogitsLoss()(tb, one)                  # the loss takes the logit directly
    ga = torch.autograd.grad(La, ta)[0].item(); gb = torch.autograd.grad(Lb, tb)[0].item()
    rows.append((zv, La.item(), Lb.item(), ga, gb))
print(f"\n{'z':>7}{'BCELoss(sig(z))':>17}{'WithLogits(z)':>15}{'exact -log sig':>16}{'grad, trap':>12}{'grad, logits':>14}")
for zv, La, Lb, ga, gb in rows:
    print(f"{zv:>7.0f}{La:>17.3f}{Lb:>15.3f}{neg_log_sig_stable(zv) if neg_log_sig_stable(zv) is not None else float('nan'):>16.3f}{ga:>12.3f}{gb:>14.3f}")

trap_gradient_at_minus_100 = None                         # YOUR CODE HERE: read it off the table (a number)
logits_gradient_at_minus_100 = None                       # YOUR CODE HERE

In [ ]:
check_close("B3 sigmoid", sigmoid(np.array([-2.0, 0.0, 3.0])), 1 / (1 + np.exp(-np.array([-2.0, 0.0, 3.0]))), tol=1e-9)
check_close("B3 bce_from_logits matches BCEWithLogitsLoss", ours, torch_val, tol=1e-4,
            hint="-sum((1-y) log(1-lam) + y log lam) with lam = sigmoid(z).")
check_close("B3 stable -log sig[z]", neg_log_sig_stable(np.array([-800.0, 0.0, 30.0])),
            [800.0, np.log(2), np.log1p(np.exp(-30.0))], tol=1e-6, hint="np.logaddexp(0, -z).")
check("B3 the trap's gradient at z = -100 is zero", None if trap_gradient_at_minus_100 is None else abs(trap_gradient_at_minus_100) < 1e-6,
      "sigmoid(-100) rounds to exactly 0 in float32; log(0) is clamped and the gradient dies.")
check("B3 the logit loss's gradient at z = -100 is -1", None if logits_gradient_at_minus_100 is None else abs(logits_gradient_at_minus_100 + 1) < 1e-6,
      "d/dz [log(1 + e^{-z})] = -(1 - sig[z]), which is -1 for very negative z.")

**Written.** A colleague builds a defect detector as `nn.Sequential(..., nn.Linear(64, 1), nn.Sigmoid())` and trains it with `nn.BCELoss()`. Training stalls with a few examples stuck confidently wrong. Using the table, explain in three sentences what is happening numerically, what the gradient *should* be for a confidently wrong example, and the one-line fix. (Week 5's rule: *the loss takes the logits.*)

*Your answer:*

### B4 · Softmax and the multiclass cross-entropy *(25 min)*

*Midterm: the categorical construction, softmax, log-sum-exp, cross-entropy as negative log-likelihood (Prince §5.5, §5.7).*

With $K$ classes the network has $K$ outputs (logits) $\mathbf{z}$, the softmax turns them into a categorical distribution, and the loss is $-\sum_i \log \mathrm{softmax}_{y_i}[\mathbf{z}_i] = \sum_i \big[\log \sum_k e^{z_{ik}} - z_{i y_i}\big]$.

1. Implement `softmax` (subtracting the row maximum first) and `log_softmax` via `logsumexp`.
2. Implement the cross-entropy loss and match `nn.CrossEntropyLoss(reduction="sum")`.
3. Verify that adding the same constant to every logit leaves the softmax unchanged, and that the loss for a single example is $\log K$ when all logits are equal.

In [ ]:
def softmax(Z):
    """Row-wise softmax of logits Z with shape (I, K); subtract the row max for stability."""
    return None                                           # YOUR CODE HERE

def log_softmax(Z):
    """log softmax, via the log-sum-exp trick, never forming exp(Z) alone."""
    m = Z.max(axis=1, keepdims=True)
    return None                                           # YOUR CODE HERE: Z - (m + log(sum(exp(Z - m))))

def cross_entropy(Z, y):
    """Multiclass cross-entropy summed over examples; y holds integer class labels. Equation (5.24)."""
    return None                                           # YOUR CODE HERE: -sum over i of log_softmax(Z)[i, y_i]

torch.manual_seed(774)
I_c, K_c = 8, 5
Z_t = torch.randn(I_c, K_c) * 3; y_c = torch.randint(0, K_c, (I_c,))
ours_ce = cross_entropy(Z_t.numpy(), y_c.numpy())
torch_ce = nn.CrossEntropyLoss(reduction="sum")(Z_t, y_c).item()
print(f"ours {ours_ce}   PyTorch {torch_ce:.6f}")

Z1 = np.array([[1.2, -0.4, 2.0, 0.3]])
shift_invariant = None                                    # YOUR CODE HERE: True/False: np.allclose(softmax(Z1), softmax(Z1 + 1000))
loss_equal_logits = None                                  # YOUR CODE HERE: cross_entropy(np.zeros((1, 7)), np.array([3]))
big = np.array([[1000.0, 1001.0, 999.0]])
naive_ok = bool(np.all(np.isfinite(np.exp(big) / np.exp(big).sum()))) if False else None   # leave as is
print("shift invariant:", shift_invariant, "  loss with 7 equal logits:", loss_equal_logits, " log 7 =", np.log(7))
degrees_of_freedom = None                                 # by hand: how many of the K logits carry independent information? write an expression in K like "K-1"

In [ ]:
_Z = Z_t.numpy()
_E = np.exp(_Z - _Z.max(1, keepdims=True)); _S = _E / _E.sum(1, keepdims=True)
check_close("B4 softmax rows sum to one and match", softmax(_Z), _S, tol=1e-6, hint="exp(Z - rowmax) / row sum.")
check_close("B4 log_softmax", log_softmax(_Z), np.log(_S), tol=1e-6)
check_close("B4 cross_entropy matches CrossEntropyLoss", ours_ce, torch_ce, tol=1e-4,
            hint="Pick out log_softmax(Z)[i, y_i] for each i, negate, sum.")
check("B4 softmax works on logits near 1000", None if softmax(big) is None else np.all(np.isfinite(softmax(big))),
      "Subtract the row maximum before exponentiating.")
check("B4 shift invariance", shift_invariant, "softmax(z + c) == softmax(z) for any constant c.")
check_close("B4 loss with K equal logits is log K", loss_equal_logits, np.log(7), tol=1e-6)
check_answer("B4 degrees of freedom among K logits", None if degrees_of_freedom is None else degrees_of_freedom.replace(" ", ""), '7c35c5a1785d',
             "The softmax ignores a common shift, so one direction in logit space is invisible.")

**Written.** Prince shows that the multiclass cross-entropy and the negative log-likelihood of the categorical model are the same thing (§5.7). In three sentences: what is the *cross-entropy* between the empirical label distribution $q$ and the model's $Pr(y \mid \mathbf{x})$, why is minimizing it over $\boldsymbol{\phi}$ identical to maximum likelihood, and can the training loss of a softmax classifier ever reach exactly zero? (Compare the final training loss in the Week 8 overconfidence figure.)

*Your answer:*

## Part C — Fitting models

*Week 6 note; Prince, Chapter 6.*

Training is: start somewhere, and repeatedly step against the gradient. Everything in Chapter 6 is about what can go wrong with that sentence — the step size, the shape of the surface, the cost of the gradient — and the three fixes that every modern optimizer is built from: batches, momentum, and per-parameter normalization.

### C1 · Gradient descent and the step size *(25 min)*

*Midterm: the gradient descent update, learning rate, the stability limit set by curvature (Prince §6.1; Week 6 note).*

For linear regression $L[\boldsymbol{\phi}] = \|\mathbf{X}\boldsymbol{\phi} - \mathbf{y}\|^2$ the gradient is $2\mathbf{X}^T(\mathbf{X}\boldsymbol{\phi} - \mathbf{y})$ and the Hessian is $\mathbf{H} = 2\mathbf{X}^T\mathbf{X}$. Gradient descent with a fixed learning rate $\alpha$ converges if and only if $\alpha < 2/\lambda_{\max}[\mathbf{H}]$.

1. Implement `grad_lin` and `gradient_descent`.
2. Compute $\alpha_{\text{crit}} = 2/\lambda_{\max}$ for the raw inputs, then standardize the input column and recompute it along with the condition number $\lambda_{\max}/\lambda_{\min}$ of each.
3. On the standardized problem run with $\alpha = 0.5\,\alpha_{\text{crit}}$, $0.95\,\alpha_{\text{crit}}$ and $1.05\,\alpha_{\text{crit}}$ and record which converge. Then count the steps each problem needs at $0.95\,\alpha_{\text{crit}}$.

In [ ]:
x_c = rng.uniform(20, 30, 40)                              # a raw temperature, in degrees C
y_c = 3.0 + 0.4 * x_c + rng.normal(0, 0.5, 40)
X_c = np.column_stack([np.ones(40), x_c])
phi_closed = np.linalg.solve(X_c.T @ X_c, X_c.T @ y_c)

def loss_lin(phi, X=X_c, y=y_c):  return np.sum((X @ phi - y) ** 2)
def grad_lin(phi, X=X_c, y=y_c):  return None             # YOUR CODE HERE: equation (6.7), a length-2 vector

def gradient_descent(phi0, alpha, n_steps, X=X_c, y=y_c):
    """Returns the sequence of iterates as an array of shape (n_steps + 1, 2)."""
    phi, path = np.asarray(phi0, float).copy(), [np.asarray(phi0, float).copy()]
    for _ in range(n_steps):
        pass                                               # YOUR CODE HERE: phi = phi - alpha * grad
        path.append(phi.copy())
    return np.array(path)

H_c = None                                                 # YOUR CODE HERE: the Hessian 2 X^T X
lam_c = None                                               # YOUR CODE HERE: its eigenvalues, np.linalg.eigvalsh
alpha_crit = None                                          # YOUR CODE HERE: 2 / lambda_max

# Standardize the input column (mean 0, variance 1) and redo the curvature analysis.
x_std = (x_c - x_c.mean()) / x_c.std()
X_s = np.column_stack([np.ones(40), x_std])
lam_s = None                                               # YOUR CODE HERE
alpha_crit_std = None                                      # YOUR CODE HERE
cond_raw, cond_std = None, None                            # YOUR CODE HERE: lambda_max / lambda_min for each
print("alpha_crit raw", alpha_crit, " standardized", alpha_crit_std)
print("condition number raw", cond_raw, " standardized", cond_std)

# Stability: on the standardized problem, run 200 steps at three fractions of alpha_crit_std.
phi_closed_s = np.linalg.solve(X_s.T @ X_s, X_s.T @ y_c)
converged = {}                                             # fraction of alpha_crit_std -> True/False
for frac in (0.5, 0.95, 1.05):
    pass                                                   # YOUR CODE HERE: run 200 steps from [0, 0] on (X_s, y_c); converged if |final loss - optimum| < 1e-6
print("converged:", converged)

def steps_to_converge(X, y, alpha, max_steps=300000):
    """Steps of gradient descent from [0, 0] until the loss is within 1e-6 of its minimum, or None."""
    phi_star = np.linalg.solve(X.T @ X, X.T @ y); L_star = loss_lin(phi_star, X, y)
    phi = np.zeros(2)
    for t in range(1, max_steps + 1):
        pass                                               # YOUR CODE HERE: one step; return t when loss_lin(phi, X, y) - L_star < 1e-6
    return None

steps_std = None                                           # YOUR CODE HERE: steps_to_converge(X_s, y_c, 0.95 * alpha_crit_std)
steps_raw = None                                           # YOUR CODE HERE: steps_to_converge(X_c, y_c, 0.95 * alpha_crit)
print("steps to converge at 0.95 alpha_crit: standardized", steps_std, " raw", steps_raw)

In [ ]:
_lam = np.linalg.eigvalsh(2 * X_c.T @ X_c); _ac = 2 / _lam[-1]
check_close("C1 grad_lin", grad_lin(np.array([1.0, 0.1])), 2 * X_c.T @ (X_c @ np.array([1.0, 0.1]) - y_c), tol=1e-9,
            hint="2 X^T (X phi - y).")
check_close("C1 alpha_crit", alpha_crit, _ac, tol=1e-9, hint="2 / largest eigenvalue of 2 X^T X.")
check("C1 converges below the limit, diverges above",
      None if len(converged) < 3 else converged.get(0.5) and converged.get(0.95) and not converged.get(1.05),
      "With alpha > 2/lambda_max the component along the steepest eigenvector grows every step.")
check("C1 the raw problem needs thousands of times more steps",
      None if None in (steps_std, steps_raw) else steps_std < 500 and steps_raw > 100 * steps_std,
      "At 0.95 alpha_crit the shallow direction contracts by only about 1 - 2 lambda_min / lambda_max per step.")
check_close("C1 standardized condition number is 1", cond_std, 1.0, tol=1e-6,
            hint="A centered, unit-variance column is orthogonal to the column of ones with the same norm.")
check("C1 raw condition number is huge", None if cond_raw is None else cond_raw > 1000)

**Written.** With raw temperatures the condition number is in the thousands; standardized, it is one. In two or three sentences: what does the condition number of the Hessian tell you about gradient descent with a single learning rate, why does one eigenvalue dictate the largest safe step while the other dictates how slowly the valley floor is crossed, and what does this predict about the number of steps needed in each case?

*Your answer:*

### C2 · Zero gradient is not the same as a minimum *(20 min)*

*Midterm: local minima, saddle points, the Hessian test; why non-convexity matters for deep networks (Prince §6.1.3–6.1.4; Week 6 note on saddles).*

The surface $L[\boldsymbol{\phi}] = \phi_0^4/4 - \phi_0^2/2 + \phi_1^2/2$ has three critical points: $(0, 0)$, $(1, 0)$, and $(-1, 0)$.

1. Implement the gradient and the Hessian.
2. Classify each critical point from the Hessian's eigenvalues: both positive → minimum, both negative → maximum, mixed → saddle.
3. Run gradient descent from $\boldsymbol{\phi} = [10^{-7}, 1.5]^T$ with $\alpha = 0.1$ and record the step at which $\|\nabla L\|$ first drops below $10^{-3}$ and the step at which the loss first drops below $-0.2$.

In [ ]:
def L_toy(p):   return p[0] ** 4 / 4 - p[0] ** 2 / 2 + p[1] ** 2 / 2
def g_toy(p):   return None                                # YOUR CODE HERE: gradient, length 2
def H_toy(p):   return None                                # YOUR CODE HERE: 2x2 Hessian

critical = {(0.0, 0.0): None, (1.0, 0.0): None, (-1.0, 0.0): None}    # fill with "minimum", "maximum", or "saddle"

p = np.array([1e-7, 1.5]); gnorm, losses = [], []
for t in range(400):
    pass                                                   # YOUR CODE HERE: record gnorm, losses; then p = p - 0.1 * g_toy(p)
gnorm, losses = np.array(gnorm), np.array(losses)

first_small_grad = None                                    # YOUR CODE HERE: first t with gnorm[t] < 1e-3
first_low_loss   = None                                    # YOUR CODE HERE: first t with losses[t] < -0.2
print("gradient first below 1e-3 at step", first_small_grad, "; loss first below -0.2 at step", first_low_loss)

if len(gnorm):
    fig, ax = plt.subplots(1, 2, figsize=(8, 3))
    ax[0].semilogy(gnorm, color=NDSU_GREEN); ax[0].axhline(1e-3, ls=":", color="gray"); ax[0].set(xlabel="step", ylabel=r"$\|\nabla L\|$")
    ax[1].plot(losses, color=ACCENT); ax[1].set(xlabel="step", ylabel="loss"); plt.tight_layout(); plt.show()

In [ ]:
_g = lambda p: np.array([p[0] ** 3 - p[0], p[1]])
_H = lambda p: np.array([[3 * p[0] ** 2 - 1, 0.0], [0.0, 1.0]])
check_close("C2 gradient", g_toy(np.array([0.5, -2.0])), _g(np.array([0.5, -2.0])), tol=1e-9, hint="[p0^3 - p0, p1].")
check_close("C2 Hessian", H_toy(np.array([0.5, -2.0])), _H(np.array([0.5, -2.0])), tol=1e-9, hint="diag(3 p0^2 - 1, 1).")
check_answer("C2 classification of the origin", critical[(0.0, 0.0)], 'ba08dad18ac8', "Eigenvalues -1 and +1.")
check_answer("C2 classification of (1, 0)", critical[(1.0, 0.0)], '692e4e5decff', "Eigenvalues 2 and 1.")
check_answer("C2 classification of (-1, 0)", critical[(-1.0, 0.0)], '692e4e5decff')
_p = np.array([1e-7, 1.5]); _gn, _ls = [], []
for _ in range(400):
    _gn.append(np.linalg.norm(_g(_p))); _ls.append(L_toy(_p)); _p = _p - 0.1 * _g(_p)
check_close("C2 step at which the gradient first looks converged", first_small_grad, int(np.argmax(np.array(_gn) < 1e-3)), tol=0)
check_close("C2 step at which the loss actually drops", first_low_loss, int(np.argmax(np.array(_ls) < -0.2)), tol=0)
check("C2 the gradient test fires long before the real descent", None if None in (first_small_grad, first_low_loss) else first_small_grad < first_low_loss - 20)

**Written.** A stopping rule "halt when $\|\nabla L\| < 10^{-3}$" would have stopped this run at a point with loss 0, missing the minimum at loss $-0.25$. In three sentences: what is a saddle point, why does Prince argue that saddles rather than local minima are the typical obstacle in high-dimensional loss surfaces, and why does the tiny $10^{-7}$ offset eventually rescue the run?

*Your answer:*

### C3 · Stochastic gradient descent: noise, batches, schedules *(30 min)*

*Midterm: minibatches and epochs, why SGD's noise is a feature, the noise floor and learning rate schedules (Prince §6.2; Week 6 note).*

Back to the standardized linear regression of C1. A batch gradient $\frac{I}{B}\sum_{i \in \mathcal{B}} \nabla \ell_i$ is an unbiased but noisy estimate of the full gradient.

1. Implement `batch_gradient` (scaled by $I/B$ so it is comparable with the full gradient) and measure, at a point near the minimum, the fraction of random batches whose step points **uphill** on the full loss, for several batch sizes.
2. Implement one epoch of SGD — shuffle, split into batches, one update per batch.
3. Run 60 epochs with (a) a constant $\alpha = 0.1$, (b) a constant $\alpha = 0.01$, and (c) $\alpha = 0.1$ halved every 10 epochs. Record the final excess loss $L[\boldsymbol{\phi}] - L[\hat{\boldsymbol{\phi}}]$ of each.

In [ ]:
I_s = 40
phi_hat_s = np.linalg.solve(X_s.T @ X_s, X_s.T @ y_c)
L_min_s = loss_lin(phi_hat_s, X_s, y_c)

def batch_gradient(phi, idx):
    """(I / B) * gradient of the loss over the examples in idx."""
    return None                                            # YOUR CODE HERE: 2 * X_s[idx].T @ (X_s[idx] @ phi - y_c[idx]) * I_s / len(idx)

phi_probe = phi_hat_s + np.array([0.3, 0.25])            # close to the minimum, where batch gradients disagree
g_full = grad_lin(phi_probe, X_s, y_c)
uphill = {}                                                # batch size -> fraction of 3000 random batches with g_batch . g_full < 0
for B in (1, 2, 5, 10, 20, 40):
    pass                                                   # YOUR CODE HERE: idx = rng.choice(I_s, B, replace=False)
print("fraction of uphill batch steps:", {B: round(v, 3) for B, v in uphill.items()})

def sgd(alpha_at_epoch, n_epochs=60, B=2, seed=0):
    """Minibatch SGD from [0, 0]. alpha_at_epoch(e) gives the learning rate for epoch e. Returns the excess loss per epoch."""
    g = np.random.default_rng(seed)
    phi, hist = np.zeros(2), []
    for e in range(n_epochs):
        perm = g.permutation(I_s)
        for b in range(0, I_s, B):
            pass                                           # YOUR CODE HERE: idx = perm[b:b+B]; phi = phi - alpha_at_epoch(e) * batch_gradient(phi, idx) / I_s
        hist.append(loss_lin(phi, X_s, y_c) - L_min_s)
    return np.array(hist)

runs = {"constant 0.1":   None,                            # YOUR CODE HERE: np.median over seeds 0..9 of sgd(lambda e: 0.1, seed=s)
        "constant 0.01":  None,
        "0.1 halved every 10": None}
for name, h in runs.items():
    if h is not None: print(f"{name:<22} final excess loss {h[-1]:.2e}")

if all(h is not None for h in runs.values()):
    plt.figure(figsize=(6.5, 3.2))
    for (name, h), c in zip(runs.items(), (ACCENT, STEEL, NDSU_GREEN)):
        plt.semilogy(h, color=c, label=name)
    plt.xlabel("epoch"); plt.ylabel("excess loss"); plt.legend(); plt.show()

In [ ]:
_bg = lambda phi, idx: 2 * X_s[idx].T @ (X_s[idx] @ phi - y_c[idx]) * I_s / len(idx)
check_close("C3 batch_gradient scaling", batch_gradient(phi_probe, np.arange(5)), _bg(phi_probe, np.arange(5)), tol=1e-9,
            hint="Gradient over the batch, times I / B.")
check_close("C3 a full batch is the full gradient", None if batch_gradient(phi_probe, np.arange(I_s)) is None else batch_gradient(phi_probe, np.arange(I_s)), g_full, tol=1e-9)
check("C3 uphill fraction falls with batch size and is zero at B = I",
      None if len(uphill) < 6 else uphill[1] > uphill[5] > uphill[20] and uphill[40] == 0.0,
      "Larger batches average out the noise; with B = I there is none.")
_ok = all(h is not None for h in runs.values())
check("C3 three runs recorded", None if not _ok else all(len(h) == 60 for h in runs.values()))
if _ok:
    check("C3 the large constant rate stalls at a noise floor above the small one",
          runs["constant 0.1"][-1] > 3 * runs["constant 0.01"][-1],
          "A constant large step keeps bouncing by an amount set by batch-to-batch disagreement.")
    check("C3 the schedule beats both constants", runs["0.1 halved every 10"][-1] < runs["constant 0.01"][-1],
          "Fast early, precise late.")
    check("C3 the small constant rate is slow early", runs["constant 0.01"][3] > runs["constant 0.1"][3])

**Written.** Prince lists three properties of SGD: it is cheaper per step, it can escape local minima and saddles, and it tends to find solutions that generalize well. In three sentences: which of the three does the uphill-fraction table illustrate, what is the *noise floor* seen in the constant-0.1 run and what sets its height, and why does a decaying schedule remove it without giving up the early speed?

*Your answer:*

### C4 · Momentum and Adam, by hand and in PyTorch *(35 min)*

*Midterm: the momentum recursion and what it does on an ill-conditioned bowl; Adam's normalization and bias correction; PyTorch's conventions (Prince §6.3–6.4; Week 6 note).*

On the bowl $L = \tfrac{1}{2}(0.1\,\phi_0^2 + 10\,\phi_1^2)$ gradient descent must keep $\alpha < 2/10$ because of the steep axis, and then crawls along the shallow one. Prince's updates:

- **Momentum** (6.11–6.12): $\mathbf{m}_{t+1} = \beta\,\mathbf{m}_t + (1-\beta)\,\nabla L$, then $\boldsymbol{\phi}_{t+1} = \boldsymbol{\phi}_t - \alpha\,\mathbf{m}_{t+1}$.
- **Adam** (6.13–6.16): $\mathbf{m} \leftarrow \beta\mathbf{m} + (1-\beta)\nabla L$, $\mathbf{v} \leftarrow \gamma\mathbf{v} + (1-\gamma)(\nabla L)^2$, bias-correct $\tilde{\mathbf{m}} = \mathbf{m}/(1-\beta^{t})$, $\tilde{\mathbf{v}} = \mathbf{v}/(1-\gamma^{t})$, then $\boldsymbol{\phi} \leftarrow \boldsymbol{\phi} - \alpha\,\tilde{\mathbf{m}}/(\sqrt{\tilde{\mathbf{v}}} + \epsilon)$.

1. Implement the three updates and run each for 150 steps from $[-5, 1]^T$ with the learning rates given.
2. Match your momentum run to `torch.optim.SGD`, remembering that PyTorch's `momentum=β` with `lr` corresponds to Prince's $\alpha = $ `lr`$/(1-\beta)$.
3. Check the bias correction: for a constant gradient $g$, the corrected ratio $\tilde{m}/\sqrt{\tilde{v}}$ should equal exactly 1 at every step.

In [ ]:
A_bowl = np.diag([0.1, 10.0])
L_bowl = lambda p: 0.5 * p @ A_bowl @ p
g_bowl = lambda p: A_bowl @ p
p_start = np.array([-5.0, 1.0])

def run(update, n=150):
    phi, state, path = p_start.copy(), {}, [p_start.copy()]
    for t in range(1, n + 1):
        phi = update(phi, g_bowl(phi), state, t)
        if phi is None:
            return None                                    # the update is still blank
        path.append(phi.copy())
    return np.array(path)

def gd_update(phi, g, state, t, alpha=0.19):
    return None                                            # YOUR CODE HERE

def momentum_update(phi, g, state, t, alpha=1.5, beta=0.9):
    m = state.get("m", np.zeros_like(phi))
    return None                                            # YOUR CODE HERE: update m, store it in state["m"], step

def adam_update(phi, g, state, t, alpha=0.3, beta=0.9, gamma=0.999, eps=1e-8):
    m = state.get("m", np.zeros_like(phi)); v = state.get("v", np.zeros_like(phi))
    return None                                            # YOUR CODE HERE: equations (6.13)-(6.16)

paths = {"GD": run(gd_update), "momentum": run(momentum_update), "Adam": run(adam_update)}
for name, path in paths.items():
    if path is not None: print(f"{name:<10} distance to minimum after 150 steps: {np.linalg.norm(path[-1]):.2e}")

# PyTorch's momentum SGD with the matching convention.
w = torch.tensor(p_start, requires_grad=True)
opt = None                                                 # YOUR CODE HERE: torch.optim.SGD([w], lr=..., momentum=0.9) matching alpha=1.5, beta=0.9
torch_path = [p_start.copy()]
for _ in range(150):
    if opt is None: break
    opt.zero_grad(); (0.5 * w @ torch.tensor(A_bowl) @ w).backward(); opt.step()
    torch_path.append(w.detach().numpy().copy())
torch_path = np.array(torch_path)
max_gap = None if opt is None or paths["momentum"] is None else np.abs(torch_path - paths["momentum"]).max()
print("max |PyTorch - ours| over the momentum path:", max_gap)

# Bias correction with a constant gradient.
ratio_corrected = []
m = v = 0.0; g_const = 0.37
for t in range(1, 6):
    pass                                                   # YOUR CODE HERE: update m, v; append m_tilde / sqrt(v_tilde)
print("corrected m~/sqrt(v~) for a constant gradient:", np.round(ratio_corrected, 6))

if all(p is not None for p in paths.values()):
    th = np.linspace(0, 2 * np.pi, 200)
    plt.figure(figsize=(6.5, 3.6))
    for lv in (0.5, 2, 5, 12):
        plt.plot(np.sqrt(2 * lv / 0.1) * np.cos(th), np.sqrt(2 * lv / 10) * np.sin(th), color="gray", lw=0.5)
    for (name, path), c in zip(paths.items(), (ACCENT, STEEL, NDSU_GREEN)):
        plt.plot(path[:40, 0], path[:40, 1], "-o", ms=2.5, lw=1, color=c, label=name)
    plt.plot(0, 0, "k*", ms=10); plt.legend(); plt.xlabel(r"$\phi_0$"); plt.ylabel(r"$\phi_1$"); plt.ylim(-2.5, 2.5); plt.show()

In [ ]:
def _ref_run(kind, n=150):
    phi, m, v, path = p_start.copy(), np.zeros(2), np.zeros(2), [p_start.copy()]
    for t in range(1, n + 1):
        g = A_bowl @ phi
        if kind == "GD": phi = phi - 0.19 * g
        elif kind == "momentum": m = 0.9 * m + 0.1 * g; phi = phi - 1.5 * m
        else:
            m = 0.9 * m + 0.1 * g; v = 0.999 * v + 0.001 * g ** 2
            phi = phi - 0.3 * (m / (1 - 0.9 ** t)) / (np.sqrt(v / (1 - 0.999 ** t)) + 1e-8)
        path.append(phi.copy())
    return np.array(path)
for k in ("GD", "momentum", "Adam"):
    check_close(f"C4 {k} path", paths[k], _ref_run(k), tol=1e-6,
                hint="Follow the equations exactly; momentum uses (1 - beta) g, Adam divides by (1 - beta^t) and (1 - gamma^t).")
_d = {k: np.linalg.norm(_ref_run(k)[-1]) for k in ("GD", "momentum", "Adam")}
check("C4 GD is slowest on the ill-conditioned bowl", _d["GD"] > _d["momentum"] and _d["GD"] > _d["Adam"])
check("C4 PyTorch momentum SGD matches ours", None if max_gap is None else max_gap < 1e-4,
      "torch.optim.SGD(lr = alpha * (1 - beta), momentum = beta) reproduces Prince's recursion.")
check_close("C4 bias-corrected ratio is exactly one", None if len(ratio_corrected) < 5 else ratio_corrected, np.ones(5), tol=1e-9,
            hint="With a constant gradient m_t = (1 - beta^t) g and v_t = (1 - gamma^t) g^2; correcting both gives g / |g| = 1.")

**Written.** Three sentences: (i) why can momentum take a learning rate seven times larger than plain gradient descent on this bowl without diverging — what happens to the alternating gradients along $\phi_1$ and the aligned ones along $\phi_0$? (ii) What does Adam's division by $\sqrt{\tilde{v}}$ do to the step along each axis, and what would go wrong in the first few iterations without bias correction? (iii) A colleague switches `torch.optim.SGD(lr=0.05)` to `torch.optim.SGD(lr=0.05, momentum=0.9)` and training diverges — explain using the convention you verified.

*Your answer:*

## Part D — Gradients and initialization

*Week 7 note; Prince, Chapter 7.*

Two problems that are specific to deep networks: computing the gradient of the loss with respect to every parameter cheaply, and choosing where to start so that signals neither vanish nor explode on the way through the layers.

### D1 · Backpropagation by hand on the toy model *(30 min)*

*Midterm: the forward pass, backward pass 1 (intermediate derivatives), backward pass 2 (parameter derivatives), the two observations behind the algorithm (Prince §7.2–7.3).*

Prince's toy model, $\mathrm{f}[x, \boldsymbol{\phi}] = \beta_3 + \omega_3 \cos[\beta_2 + \omega_2 \exp[\beta_1 + \omega_1 \sin[\beta_0 + \omega_0 x]]]$ with loss $\ell = (\mathrm{f} - y)^2$. Every intermediate value is a scalar, so there is nowhere for the idea to hide.

1. **Forward pass:** compute and store $f_0, h_1, f_1, h_2, f_2, h_3, f_3$ and the loss.
2. **Backward pass 1:** compute $\partial\ell/\partial f_3, \partial\ell/\partial h_3, \ldots, \partial\ell/\partial f_0$, each line reusing the one before.
3. **Backward pass 2:** the eight parameter derivatives, then check all of them against central finite differences and against PyTorch autograd.

In [ ]:
beta  = np.array([0.5, -0.3, 0.2, 1.0])
omega = np.array([1.1,  0.7, -0.6, 0.9])
x_i, y_i = 2.3, 1.4

def forward(beta, omega, x, y):
    """Equation (7.8): compute and store every intermediate value."""
    f0 = None; h1 = None                                   # YOUR CODE HERE
    f1 = None; h2 = None
    f2 = None; h3 = None
    f3 = None
    return dict(f0=f0, h1=h1, f1=f1, h2=h2, f2=f2, h3=h3, f3=f3, loss=None)

s = forward(beta, omega, x_i, y_i)

# Backward pass 1, equations (7.10)-(7.12). Each line is the previous one times one local derivative.
dl_df3 = None                                              # YOUR CODE HERE: 2 (f3 - y)
dl_dh3 = None                                              # df3/dh3 = omega_3
dl_df2 = None                                              # dh3/df2 = -sin[f2]
dl_dh2 = None
dl_df1 = None                                              # dh2/df1 = exp[f1]
dl_dh1 = None
dl_df0 = None                                              # dh1/df0 = cos[f0]

# Backward pass 2, equations (7.13)-(7.15).
dl_dbeta  = None                                           # YOUR CODE HERE: array of the four dl/dbeta_k
dl_domega = None                                           # YOUR CODE HERE: array of the four dl/domega_k

# Finite differences and autograd for comparison.
eps = 1e-6
def fd(which, k):
    e = np.zeros(4); e[k] = eps
    if which == "beta":
        return (forward(beta + e, omega, x_i, y_i)["loss"] - forward(beta - e, omega, x_i, y_i)["loss"]) / (2 * eps)
    return (forward(beta, omega + e, x_i, y_i)["loss"] - forward(beta, omega - e, x_i, y_i)["loss"]) / (2 * eps)

tb = torch.tensor(beta, requires_grad=True); tw = torch.tensor(omega, requires_grad=True)
t_f = tb[3] + tw[3] * torch.cos(tb[2] + tw[2] * torch.exp(tb[1] + tw[1] * torch.sin(tb[0] + tw[0] * x_i)))
((t_f - y_i) ** 2).backward()

if dl_dbeta is not None and s["loss"] is not None:
    print(f"{'parameter':<10}{'backprop':>13}{'finite diff.':>14}{'autograd':>13}")
    for k in range(4):
        print(f"{'beta_' + str(k):<10}{dl_dbeta[k]:>13.8f}{fd('beta', k):>14.8f}{tb.grad[k].item():>13.8f}")
        print(f"{'omega_' + str(k):<10}{dl_domega[k]:>13.8f}{fd('omega', k):>14.8f}{tw.grad[k].item():>13.8f}")

# Which stored forward value does each weight derivative multiply? (Observation 1)
source_of_omega2 = None                                    # "x", "h1", "h2", or "h3"

In [ ]:
check_close("D1 forward pass loss", s["loss"], ((t_f - y_i) ** 2).item(), tol=1e-9,
            hint="f0 = b0 + w0 x, h1 = sin f0, f1 = b1 + w1 h1, h2 = exp f1, f2 = b2 + w2 h2, h3 = cos f2, f3 = b3 + w3 h3.")
check_close("D1 dl/df3", dl_df3, 2 * (t_f.item() - y_i), tol=1e-9)
check_close("D1 dl/df0 (end of backward pass 1)", dl_df0, tb.grad[0].item(), tol=1e-7,
            hint="dl/dbeta_0 equals dl/df0 because f0 = beta_0 + omega_0 x.")
check_close("D1 all eight parameter gradients match autograd",
            None if dl_dbeta is None or dl_domega is None else np.r_[dl_dbeta, dl_domega],
            np.r_[tb.grad.numpy(), tw.grad.numpy()], tol=1e-7,
            hint="dl/dbeta_k = dl/df_k; dl/domega_k = dl/df_k times the source value (x or h_k).")
check_close("D1 finite differences agree too", None if dl_dbeta is None else [fd("beta", k) for k in range(4)],
            tb.grad.numpy(), tol=1e-5)
check_answer("D1 source value multiplying dl/domega_2", source_of_omega2, 'f998fe06afa0',
             "omega_2 multiplies h2 in f2 = beta_2 + omega_2 h2.")

**Written.** Prince builds the algorithm on two observations. State both in your own words and point to the exact line of your code where each one appears. Then explain why computing the eight derivatives this way is cheaper than differentiating the full expression for $\ell$ with respect to each parameter separately.

*Your answer:*

### D2 · Backpropagation in matrix form *(35 min)*

*Midterm: the matrix backward pass — transpose, mask, outer product — and the shape of every term (Prince §7.4).*

For a deep ReLU network, backward pass 1 alternates **multiplying by a transposed weight matrix** and **masking by which pre-activations were positive**, and backward pass 2 forms **outer products** with the stored activations:
$$\frac{\partial\ell}{\partial\boldsymbol{\beta}_k} = \frac{\partial\ell}{\partial\mathbf{f}_k}, \qquad \frac{\partial\ell}{\partial\boldsymbol{\Omega}_k} = \frac{\partial\ell}{\partial\mathbf{f}_k}\mathbf{h}_k^T, \qquad \frac{\partial\ell}{\partial\mathbf{f}_{k-1}} = \mathbb{I}[\mathbf{f}_{k-1} > 0] \odot \Big(\boldsymbol{\Omega}_k^T \frac{\partial\ell}{\partial\mathbf{f}_k}\Big).$$

1. Fill in the three lines of the backward loop in `backprop`, for a whole batch at once (rows are examples, so the transposes move around — the comments say where).
2. Compare every gradient with PyTorch's.
3. Give the shapes, by hand, of $\partial\ell/\partial\mathbf{f}_2$, $\partial\mathbf{f}_3/\partial\mathbf{h}_3$, and $\partial\ell/\partial\boldsymbol{\Omega}_1$ for the widths used.

In [ ]:
def init_network(D, seed=774):
    """He-initialized weights and small biases for layer widths D = [D_i, D_1, ..., D_K, D_o]."""
    g = np.random.default_rng(seed)
    Omegas = [g.normal(0, np.sqrt(2 / D[k]), (D[k + 1], D[k])) for k in range(len(D) - 1)]
    betas  = [g.normal(0, 0.1, D[k + 1]) for k in range(len(D) - 1)]
    return betas, Omegas

def backprop(betas, Omegas, X, Y):
    """Equations (7.23)-(7.25) for a batch. Rows of X are inputs x_i, rows of Y targets.
    Loss is sum_i ||f_K - y_i||^2. Returns gradients summed over the batch, and the loss."""
    K = len(Omegas) - 1
    f, h = [X @ Omegas[0].T + betas[0]], [X]               # forward pass: store every f_k and h_k
    for k in range(1, K + 1):
        h.append(np.maximum(f[k - 1], 0))
        f.append(h[k] @ Omegas[k].T + betas[k])
    dl_df = 2 * (f[K] - Y)                                  # dl/df_K for least squares; one row per example
    g_beta, g_Omega = [None] * (K + 1), [None] * (K + 1)
    for k in range(K, 0, -1):
        g_beta[k]  = None                                   # YOUR CODE HERE: sum of dl/df_k over the batch (axis 0)
        g_Omega[k] = None                                   # YOUR CODE HERE: sum_i (dl/df_k)_i h_k,i^T  ->  one matrix product, dl_df.T @ h[k]
        dl_df      = None                                   # YOUR CODE HERE: mask (f[k-1] > 0) times (dl_df @ Omegas[k])   [rows are examples, so Omega^T acts as @ Omegas[k]]
        if dl_df is None:
            return g_beta, g_Omega, np.sum((f[K] - Y) ** 2)  # still blank
    g_beta[0], g_Omega[0] = dl_df.sum(axis=0), dl_df.T @ X
    return g_beta, g_Omega, np.sum((f[K] - Y) ** 2)

D = [5, 40, 30, 20, 3]                                      # D_i = 5, three hidden layers, D_o = 3
betas, Omegas = init_network(D)
X_b, Y_b = rng.normal(size=(16, 5)), rng.normal(size=(16, 3))
g_beta, g_Omega, L_ours = backprop(betas, Omegas, X_b, Y_b)

t_betas  = [torch.tensor(b, requires_grad=True) for b in betas]
t_Omegas = [torch.tensor(W, requires_grad=True) for W in Omegas]
h_t = torch.tensor(X_b)
for k, (b, W) in enumerate(zip(t_betas, t_Omegas)):
    f_t = h_t @ W.T + b
    h_t = torch.relu(f_t) if k < len(Omegas) - 1 else f_t
L_torch = ((h_t - torch.tensor(Y_b)) ** 2).sum(); L_torch.backward()

if g_Omega[1] is not None:
    print(f"loss: ours {L_ours:.8f}   PyTorch {L_torch.item():.8f}")
    for k in range(len(Omegas)):
        eW = np.abs(g_Omega[k] - t_Omegas[k].grad.numpy()).max()
        eb = np.abs(g_beta[k] - t_betas[k].grad.numpy()).max()
        print(f"layer {k}: dl/dOmega shape {g_Omega[k].shape}, max abs error {eW:.1e} (Omega) {eb:.1e} (beta)")

# Shapes, by hand, for ONE example (column-vector convention of the book), with D above.
shape_dl_df2   = None                                       # e.g. (20, 1)
shape_df3_dh3  = None                                       # the matrix d f_3 / d h_3, Prince's layout (inputs index rows)
shape_dl_dOmega1 = None                                     # same shape as Omega_1

In [ ]:
check_close("D2 loss matches", L_ours, L_torch.item(), tol=1e-9)
_ok = all(g is not None for g in g_Omega)
check("D2 every weight gradient matches PyTorch",
      None if not _ok else all(np.allclose(g_Omega[k], t_Omegas[k].grad.numpy(), atol=1e-9) for k in range(len(Omegas))),
      "g_Omega[k] = dl_df.T @ h[k]; the next dl_df = (f[k-1] > 0) * (dl_df @ Omegas[k]).")
check("D2 every bias gradient matches PyTorch",
      None if not _ok else all(np.allclose(g_beta[k], t_betas[k].grad.numpy(), atol=1e-9) for k in range(len(Omegas))),
      "g_beta[k] = dl_df.sum(axis=0): the bias gradient is dl/df_k summed over the batch.")
check_answer("D2 shape of dl/df_2", None if shape_dl_df2 is None else tuple(shape_dl_df2), '52d5d4c714b3',
             "f_2 is the pre-activation of the third hidden layer, which has D_3 = 20 units.")
check_answer("D2 shape of df_3/dh_3", None if shape_df3_dh3 is None else tuple(shape_df3_dh3), '7b23c389251d',
             "Prince's layout: rows index the input h_3 (20), columns the output f_3 (3). It equals Omega_3^T.")
check_answer("D2 shape of dl/dOmega_1", None if shape_dl_dOmega1 is None else tuple(shape_dl_dOmega1), 'bc0e99006b14',
             "Same shape as Omega_1, which maps 40 units to 30.")

**Written.** Three sentences: why is $\partial\mathbf{h}_k/\partial\mathbf{f}_{k-1}$ a *diagonal* matrix for any element-wise activation, and what is on its diagonal for the ReLU? Why does the batch version of the outer product $\sum_i (\partial\ell_i/\partial\mathbf{f}_k)\mathbf{h}_{k,i}^T$ collapse into the single product `dl_df.T @ h[k]`? And what would the backward pass look like for a sigmoid activation instead of the ReLU — what replaces the mask, and why does that cause trouble in deep networks?

*Your answer:*

### D3 · Why backprop is cheap in time and expensive in memory *(20 min)*

*Midterm: the cost of a gradient relative to a forward pass; what must be stored and why; finite differences as the alternative (Prince §7.4.1 and notes).*

1. Time one forward pass and one forward-plus-backward pass for networks of growing width, and compute their ratio.
2. Count the floats that must be stored between the forward and backward passes for a batch of $B$ examples — every pre-activation $\mathbf{f}_k$ and activation $\mathbf{h}_k$ — and compare with the number of parameters.
3. Estimate how many forward passes central finite differences would need for the same gradient.

In [ ]:
def timed(fn, reps=20):
    fn()                                                   # warm up
    t = time.perf_counter()
    for _ in range(reps): fn()
    return (time.perf_counter() - t) / reps

rows_D3 = []
for W in (64, 256, 1024):
    torch.manual_seed(774)
    net = nn.Sequential(nn.Linear(64, W), nn.ReLU(), nn.Linear(W, W), nn.ReLU(), nn.Linear(W, 10))
    xb = torch.randn(128, 64)
    def fwd():
        with torch.no_grad(): net(xb)
    def fwd_bwd():
        pass                                               # YOUR CODE HERE: net.zero_grad(); net(xb).sum().backward()
    t_f, t_fb = timed(fwd), timed(fwd_bwd)
    P = sum(p.numel() for p in net.parameters())
    rows_D3.append((W, P, t_f, t_fb, t_fb / t_f))
print(f"{'width':>6}{'params':>10}{'forward (ms)':>14}{'fwd+bwd (ms)':>14}{'ratio':>7}")
for W, P, t_f, t_fb, r in rows_D3:
    print(f"{W:>6}{P:>10}{1e3 * t_f:>14.3f}{1e3 * t_fb:>14.3f}{r:>7.2f}")

def stored_floats(D, B):
    """Pre-activations f_0..f_K and activations h_1..h_K kept for the backward pass, for a batch of B.
    D = [D_i, D_1, ..., D_K, D_o]; the output f_K is also stored (it is needed for dl/df_K)."""
    return None                                            # YOUR CODE HERE

def n_parameters(D):
    return None                                            # YOUR CODE HERE: sum over layers of D[k+1] * D[k] + D[k+1]

D_big = [784, 512, 512, 512, 10]
for B in (1, 128, 4096):
    print(f"batch {B:>5}: stored floats {stored_floats(D_big, B)}, parameters {n_parameters(D_big)}")

fd_forward_passes = None                                   # YOUR CODE HERE: central finite differences need how many forward passes for P = n_parameters(D_big)?

In [ ]:
check("D3 forward+backward timed", None if any(r[4] < 0.05 for r in rows_D3) else all(0.5 < r[4] < 20 for r in rows_D3),
      "fwd_bwd must call backward(); the ratio should be a small constant, not zero and not hundreds.")
_P = sum(D_big[k + 1] * D_big[k] + D_big[k + 1] for k in range(len(D_big) - 1))
check_close("D3 parameter count", n_parameters(D_big), _P, tol=0)
check_close("D3 stored floats for a batch of 4096", stored_floats(D_big, 4096), 4096 * (2 * 1536 + 10), tol=0,
            hint="Per example: every hidden pre-activation and activation (2 per hidden unit) plus the output.")
check("D3 at batch 4096 the activations outweigh the parameters", None if stored_floats(D_big, 4096) is None else stored_floats(D_big, 4096) > _P * 10)
check_close("D3 finite-difference forward passes", fd_forward_passes, 2 * _P, tol=0,
            hint="Two forward passes per parameter: f(phi + eps e_k) and f(phi - eps e_k).")

**Written.** Three sentences: why does the forward-plus-backward time stay a small constant multiple of the forward time however many parameters there are, whereas finite differences grow with the parameter count? Which quantities from the forward pass must be kept until the backward pass uses them, and why is this the memory bottleneck for large batches? What is *gradient checkpointing* trading for what?

*Your answer:*

### D4 · Initialization: keeping the variance level *(30 min)*

*Midterm: the variance recursion, vanishing and exploding activations and gradients, He initialization and why it has a 2, PyTorch's default (Prince §7.5).*

In a ReLU network with zero biases and weights of variance $\sigma^2_\Omega$, the variance of the pre-activations changes by a factor $\tfrac{1}{2} D_h \sigma^2_\Omega$ per layer (equation 7.30). He initialization sets $\sigma^2_\Omega = 2/D_h$ so that the factor is exactly one.

1. Implement `forward_variances`, which runs 1,000 standard-normal inputs through 40 layers of width $D_h = 200$ and records the variance of the pre-activations at each layer.
2. Run it for $\sigma^2_\Omega \in \{0.5/D_h,\ 2/D_h,\ 8/D_h\}$ and compare the last-layer variance with the prediction $(\tfrac{1}{2} D_h \sigma^2_\Omega)^{40}$.
3. Read off the variance PyTorch's default `nn.Linear` uses, by sampling, and predict what it does over 40 layers.

In [ ]:
D_h, K_layers, N_in = 200, 40, 1000

def forward_variances(sigma2, seed=774):
    """Variance of the pre-activations f_k at each of K_layers layers, for weights ~ N(0, sigma2), zero biases, ReLU."""
    g = np.random.default_rng(seed)
    h = g.standard_normal((N_in, D_h))
    out = []
    for k in range(K_layers):
        Omega = None                                       # YOUR CODE HERE: shape (D_h, D_h), variance sigma2
        f = None                                           # YOUR CODE HERE: pre-activations, rows are examples
        if f is None:
            return None                                    # still blank
        out.append(f.var())
        h = None                                           # YOUR CODE HERE: ReLU
    return np.array(out)

results = {}
for name, s2 in {"0.5/D": 0.5 / D_h, "He 2/D": 2 / D_h, "8/D": 8 / D_h}.items():
    v = forward_variances(s2)
    predicted = None                                       # YOUR CODE HERE: (0.5 * D_h * s2) ** K_layers
    results[name] = (None if v is None else v[-1], predicted)
    print(f"{name:<8} last-layer variance {results[name][0]}   predicted {predicted}")

# PyTorch's default initialization for nn.Linear(D_h, D_h): measure the weight variance.
torch.manual_seed(774)
lin = nn.Linear(D_h, D_h)
torch_default_var = None                                   # YOUR CODE HERE: lin.weight.var().item()
ratio_to_He = None                                         # YOUR CODE HERE: torch_default_var / (2 / D_h)
factor_per_layer_default = None                            # YOUR CODE HERE: 0.5 * D_h * torch_default_var
print(f"PyTorch default weight variance {torch_default_var}, which is {ratio_to_He} x He; per-layer factor {factor_per_layer_default}")

if all(v is not None for v, _ in results.values()):
    plt.figure(figsize=(6.5, 3.2))
    for (name, s2), c in zip({"0.5/D": 0.5 / D_h, "He 2/D": 2 / D_h, "8/D": 8 / D_h}.items(), (STEEL, NDSU_GREEN, ACCENT)):
        plt.semilogy(range(1, K_layers + 1), forward_variances(s2), color=c, label=name)
    plt.xlabel("layer $k$"); plt.ylabel(r"var$[\mathbf{f}_k]$"); plt.legend(); plt.show()

In [ ]:
_ok = all(v is not None for v, _ in results.values())
if _ok:
    check("D4 He initialization keeps the variance within a factor of 10 of 1 over 40 layers", 0.1 < results["He 2/D"][0] < 10,
          "Omega ~ N(0, sqrt(sigma2)); f = h @ Omega.T; h = max(f, 0).")
    check("D4 sigma^2 = 0.5/D vanishes", results["0.5/D"][0] < 1e-15, "The factor per layer is 0.25.")
    check("D4 sigma^2 = 8/D explodes", results["8/D"][0] > 1e15, "The factor per layer is 4.")
    check_close("D4 predicted factors", [results[k][1] for k in ("0.5/D", "He 2/D", "8/D")], [0.25 ** 40, 1.0, 4.0 ** 40], tol=1e-6,
                hint="(D_h sigma^2 / 2) ** K.")
else:
    check("D4 variances recorded", None)
check_close("D4 PyTorch default weight variance is about 1/(3 D)", torch_default_var, 1 / (3 * D_h), tol=0.08,
            hint="nn.Linear draws weights uniformly on [-1/sqrt(D), 1/sqrt(D)], whose variance is 1/(3D).")
check_close("D4 default is one sixth of He", ratio_to_He, 1 / 6, tol=0.08)
check("D4 default shrinks the variance each layer", None if factor_per_layer_default is None else factor_per_layer_default < 0.25)

**Written.** Three sentences: derive in words where the factor $\tfrac{1}{2}$ in equation (7.30) comes from and why He initialization therefore needs the 2 in $2/D_h$; explain what happens to the *gradients* in the backward pass under the same recursion and what that means for the early layers of a 40-layer network initialized with PyTorch's default; and say what you would check first if a deep ReLU network's training loss sat at the variance of the targets and never moved.

*Your answer:*

## Part E — Self-assessment

### E1 · Where you stand before the midterm *(10 min)*

Run the summary, then rate yourself honestly on the concepts below. Each row is one thing the midterm asks about; the task that prepares it is in the last column.

In [ ]:
summary()

For each concept, replace `?` with **3** (could teach it), **2** (can do it with the notes open), or **1** (not yet).

| # | Concept | Rating | Task |
|:--|:--|:--:|:--|
| 1 | Training loss versus test loss; underfitting, overfitting, generalization | ? | A1 |
| 2 | Joints, regions, activation patterns; why the activation must be nonlinear | ? | A2 |
| 3 | Parameter counts and region counts; representability versus trainability | ? | A3 |
| 4 | Likelihood, log-likelihood, negative log-likelihood | ? | B1 |
| 5 | Least squares as a normal model; why $\sigma^2$ drops out; what changes with other noise models | ? | B2 |
| 6 | Sigmoid, binary cross-entropy, why the loss takes logits | ? | B3 |
| 7 | Softmax, log-sum-exp, multiclass cross-entropy, cross-entropy = NLL | ? | B4 |
| 8 | Learning rate, curvature, the stability limit, conditioning | ? | C1 |
| 9 | Critical points: minima, maxima, saddles; the Hessian test | ? | C2 |
| 10 | Minibatches, epochs, SGD noise, the noise floor, schedules | ? | C3 |
| 11 | Momentum and Adam: the recursions, what they fix, PyTorch's conventions | ? | C4 |
| 12 | Forward pass and the two backward passes on the toy model; the two observations | ? | D1 |
| 13 | Matrix backprop: transpose, mask, outer product, shapes | ? | D2 |
| 14 | Cost in time versus memory; finite differences; checkpointing | ? | D3 |
| 15 | Variance recursion, vanishing/exploding, He initialization, PyTorch's default | ? | D4 |

**Muddiest point.** Which single idea from Chapters 1–7 is still least clear to you? One or two sentences; be specific. This decides what the Tuesday review session spends its time on.

*Your answer:*

**Time spent.** Roughly how many hours did this take? (One number.)

*Your answer:*

---

*IMSE 774 · Neural Networks · North Dakota State University · Fall 2026. Course notes are licensed CC BY-NC-SA 4.0; code is MIT. Equation numbers refer to Prince, Understanding Deep Learning (MIT Press, 2024), Chapters 2–7.*